In [1]:
import databento as db


In [ ]:
# Create a historical client
client = db.Historical("db-as6xhNyBf6t8HDsVXkGxfpTYgWxqJ")

BentoClientError: 422 symbology_invalid_request
None of the symbols could be resolved
documentation: https://databento.com/docs/api-reference-historical/basics/symbology

In [8]:
col_to_keep = [
    'ts_event', 'action', 'side',
       'price', 'size', 'flags', 'ts_in_delta', 
       'bid_px_00', 'ask_px_00', 'bid_sz_00', 'ask_sz_00', 'bid_ct_00',
       'ask_ct_00', 'symbol', 'mid_px'
]

In [27]:
data = client.timeseries.get_range(
    dataset="XNAS.ITCH",
    start="2023-09-29T09:30:00-04:00",
    end="2023-10-09T04:31:00-04:00",
    symbols="XOM",
    stype_in="raw_symbol",
    schema="tbbo",
)

# Convert to DataFrame
tbbo_data = data.to_df()

# Calculate the midprice for each row
tbbo_data["mid_px"] = tbbo_data[["ask_px_00", "bid_px_00"]].mean(axis=1)

result = tbbo_data[col_to_keep].copy()
result

,ts_event,action,side,price,size,flags,ts_in_delta,bid_px_00,ask_px_00,bid_sz_00,ask_sz_00,bid_ct_00,ask_ct_00,symbol,mid_px
ts_recv,,,,,,,,,,,,,,,
2023-09-29 13:30:00.172235451+00:00,2023-09-29 13:30:00.172050318+00:00,T,N,119.17,1,128,185133,119.07,119.46,100,100,1,1,XOM,119.265
2023-09-29 13:30:01.009187111+00:00,2023-09-29 13:30:01.009011016+00:00,T,N,119.12,50,0,176095,119.10,119.68,16,400,1,1,XOM,119.390
2023-09-29 13:30:01.009187111+00:00,2023-09-29 13:30:01.009011016+00:00,T,A,119.10,16,0,176095,119.10,119.68,16,400,1,1,XOM,119.390
2023-09-29 13:30:01.009228860+00:00,2023-09-29 13:30:01.009049900+00:00,T,B,119.10,134,128,178960,119.09,119.11,100,134,1,1,XOM,119.100
2023-09-29 13:30:01.009975767+00:00,2023-09-29 13:30:01.009804618+00:00,T,A,119.09,100,130,171149,119.09,119.24,100,23,1,1,XOM,119.165
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-10-09 08:28:13.423876384+00:00,2023-10-09 08:28:13.423707283+00:00,T,N,109.76,20,130,169101,109.51,109.90,300,900,1,1,XOM,109.705
2023-10-09 08:28:13.425955679+00:00,2023-10-09 08:28:13.425787953+00:00,T,A,109.77,50,130,167726,109.77,109.90,50,900,1,1,XOM,109.835
2023-10-09 08:28:54.939950484+00:00,2023-10-09 08:28:54.939782917+00:00,T,N,109.70,36,130,167567,109.69,109.90,100,900,1,1,XOM,109.795


In [28]:
result['ts_event'] = result['ts_event'].dt.tz_convert('US/Eastern')
#result['ts_recv'] = result['ts_recv'].dt.tz_convert('US/Eastern')
result

,ts_event,action,side,price,size,flags,ts_in_delta,bid_px_00,ask_px_00,bid_sz_00,ask_sz_00,bid_ct_00,ask_ct_00,symbol,mid_px
ts_recv,,,,,,,,,,,,,,,
2023-09-29 13:30:00.172235451+00:00,2023-09-29 09:30:00.172050318-04:00,T,N,119.17,1,128,185133,119.07,119.46,100,100,1,1,XOM,119.265
2023-09-29 13:30:01.009187111+00:00,2023-09-29 09:30:01.009011016-04:00,T,N,119.12,50,0,176095,119.10,119.68,16,400,1,1,XOM,119.390
2023-09-29 13:30:01.009187111+00:00,2023-09-29 09:30:01.009011016-04:00,T,A,119.10,16,0,176095,119.10,119.68,16,400,1,1,XOM,119.390
2023-09-29 13:30:01.009228860+00:00,2023-09-29 09:30:01.009049900-04:00,T,B,119.10,134,128,178960,119.09,119.11,100,134,1,1,XOM,119.100
2023-09-29 13:30:01.009975767+00:00,2023-09-29 09:30:01.009804618-04:00,T,A,119.09,100,130,171149,119.09,119.24,100,23,1,1,XOM,119.165
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-10-09 08:28:13.423876384+00:00,2023-10-09 04:28:13.423707283-04:00,T,N,109.76,20,130,169101,109.51,109.90,300,900,1,1,XOM,109.705
2023-10-09 08:28:13.425955679+00:00,2023-10-09 04:28:13.425787953-04:00,T,A,109.77,50,130,167726,109.77,109.90,50,900,1,1,XOM,109.835
2023-10-09 08:28:54.939950484+00:00,2023-10-09 04:28:54.939782917-04:00,T,N,109.70,36,130,167567,109.69,109.90,100,900,1,1,XOM,109.795


In [29]:
result.to_parquet('XOM_20230929_20231006.parquet')